In [ ]:
"""
timeseries_wm.ipynb
author: Tahya Weiss-Gibbons (adapted by Grace Kirkpatrick)

Calculate the total transport across a straight line (volume, heat, freshwater and salt)
as well as those transports for individual water masses

**IMPORTANT** 
This program calculates volume transport for a NEMO run where output temperature and salinity were calculated with respect to EOS80 rather
than the more recent TEOS10. However, as per the recommendations of McDougall et. al (2021), the output variable of NEMO is assumed to
be thermodynamically consistent with the definition of Conservative Temperature and its salinity output with Preformed Salinity / u_p. 
Calculations of density are therefore made with the gsw toolbox rather than the obsolete seawater package. 

"""

import os
os.environ['MPLCONFIGDIR'] = "/mnt/storage6/grace/plt_temp/"
import numpy as np
import matplotlib.pyplot as plt
import xarray as xr
import netCDF4 as nc
from netCDF4 import Dataset

import datetime
import cftime
import time

import gsw

In [ ]:
def define_watermasses(names, density_bounds): # list of names, list of density characteristics, creates dictionary
    watermasses = []
    for x in range(len(names)):
        watermasses.append({'name':names[x], 'rho':density_bounds[x]})
    return watermasses 

In [ ]:
def create_output_file(paths, file, wmasses): 
    
    ncfile = Dataset(paths[0] + 'wm_ts_' + file, mode='w',format='NETCDF4') 
    
    ncfile.createDimension('time_counter', None)
    time_counter = ncfile.createVariable('time_counter', int, ('time_counter',))    
    time_counter[:] = cftime.date2num(var_u[0]['time_counter'][:], 'seconds since 1900-01-01 00:00:00', calendar = 'noleap')
    #time_counter[:] = cftime.num2date(time_counter[:], 'seconds since 1900-01-01 00:00:00', calendar = 'noleap')

    dirs = ['net', 'pos', 'neg']

    for x in range(len(wmasses)):
        
        lat = ncfile.createVariable(dim_labels[x][0], float, (dim_labels[x][0], dim_labels[x][1]))
        lon = ncfile.createVariable(dim_labels[x][1], float, (dim_labels[x][0], dim_labels[x][1]))

        lat[:] = cldims[x]['nav_lat'][:,:]
        lon[:] = cldims[x]['nav_lon'][:,:]

    for x in range(len(climvars)):
        for y in range(len(var_names[0])):
            if y < 5:
                z = ncfile.createVariable(var_names[x][y], float, ('time_counter', 'depth', dim_labels[x][0], dim_labels[x][1]))
            elif y == 5:
                z = ncfile.createVariable(var_names[x][y], float, ('depth', dim_labels[x][0], dim_labels[x][1]))
            elif y == 6: 
                z = ncfile.createVariable(var_names[x][y], float, (dim_labels[x][0], dim_labels[x][1]))
            print(y)
            z[:] = climvars[x][y][:]

    ncfile.close()

In [ ]:
def get_transport(paths, file, wmasses, fwater = np.nan):

    ds = xr.open_dataset(paths[0] + file)
    
    #now we calculate the area of the cell's vertical face along both the i and j axes

    i_face = ds['e3u'][:] * ds['e1u'][:] # area of vertical cell face along i-axis, for v 
    j_face = ds['e3v'][:] * ds['e2v'][:] # area of vertical cell face alone j-axis horizontal, for u

    #now we calculate the volume flux, salinity flux, heat flux, and freshwater flux

    v_trans = (i_face * ds['v'][:] *.000001).compute() # meridional transport
    u_trans = (j_face * ds['u'][:] *.000001).compute() # zonal transport

    salv_trans = np.nansum(v_trans * sa_v, ax = 1).compute()
    salu_trans = np.nansum(u_trans * sa_u, ax = 1).compute()

    fv_trans = salv_trans - (v_trans * ref_sal) # ref_sal
    fu_trans = salv_trans - (v_trans * ref_sal) # ref_sal

    hv_trans = np.nansum(v_trans * sa_v, ax = 1).compute()
    hu_trans = np.nansum(u_trans * sa_u, ax = 1).compute()

    print(v_trans.shape)

In [ ]:
if __name__ == "__main__":
    paths = ['/mnt/storage4/grace/grace/data/transport_outputs/ANHA4/', '/mnt/storage4/grace/grace/nwcorner/Figures/']
    wmasses = define_wmasses(['surface_lc', 'ULSW', 'LLSW', 'ISOW', 'CGFZ'], [])
    tic = time.time()
    get_transport(paths, 'EPM151_preGB_2004-2005.nc', wmasses)
    toc = time.time()
    print(toc-tic)